# Результаты: 5 панельных моделей LightGBM

Каждая группа грибов — отдельная модель, обученная только на своём сезоне.
- **болетовые** (июн-окт): белый, подосиновик, подберёзовик, маслёнок, моховик
- **лисичковые** (июн-сен): лисичка обыкновенная
- **лисичковые_трубчатые** (окт-ноя): лисичка трубчатая
- **весенние** (апр-май): сморчок, строчок, сморчковая шапочка
- **опята** (июн-окт): опёнок осенний, летний

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os

preds = pd.read_csv('data/panel_predictions.csv', parse_dates=['date'])

TEST_YEAR = 2025
COLORS = {
    'болетовые': '#e6194b',
    'лисичковые': '#f58231',
    'лисичковые_трубчатые': '#ffe119',
    'весенние': '#3cb44b',
    'опята': '#4363d8',
}
GROUPS = [g for g in COLORS if g in preds['species'].unique()]

# Загружаем модели
models = {}
for g in GROUPS:
    path = f'data/panel_model_{g}.lgb'
    if os.path.exists(path):
        models[g] = lgb.Booster(model_file=path)
        print(f'  {g}: загружена')
    else:
        print(f'  {g}: модель не найдена')

train = preds[preds['date'].dt.year < TEST_YEAR]
test = preds[preds['date'].dt.year == TEST_YEAR]
print(f'\nПредсказаний: {len(preds)} (трейн: {len(train)}, тест: {len(test)})')

## 1. R2 по группам

In [ ]:
metrics = []
for g in GROUPS:
    for split_name, split_df in [('train', train), ('test', test)]:
        gd = split_df[split_df['species'] == g]
        if len(gd) == 0 or gd['mushroom_count'].std() == 0:
            continue
        r2 = r2_score(gd['mushroom_count'], gd['predicted'])
        rmse = np.sqrt(mean_squared_error(gd['mushroom_count'], gd['predicted']))
        mae = mean_absolute_error(gd['mushroom_count'], gd['predicted'])
        metrics.append({'group': g, 'split': split_name, 'R2': round(r2, 3),
                       'RMSE': round(rmse, 1), 'MAE': round(mae, 1),
                       'mean': round(gd['mushroom_count'].mean(), 1)})

mdf = pd.DataFrame(metrics)
print(mdf.to_string(index=False))

# Bar chart
test_m = mdf[mdf['split'] == 'test']
train_m = mdf[mdf['split'] == 'train']

fig = go.Figure()
fig.add_trace(go.Bar(x=test_m['group'], y=test_m['R2'],
    name='R2 test', marker_color=[COLORS[g] for g in test_m['group']],
    text=test_m['R2'], textposition='outside'))
fig.add_trace(go.Bar(x=train_m['group'], y=train_m['R2'],
    name='R2 train', marker_color='#ccc', opacity=0.5,
    text=train_m['R2'], textposition='outside'))

fig.update_layout(title='R2 по группам (test vs train)', barmode='group',
    height=450, plot_bgcolor='white', yaxis=dict(gridcolor='#e0e0e0'),
    legend=dict(orientation='h', y=-0.15))
fig.show()

## 2. Факт vs Прогноз — дневные ряды (тест)

In [ ]:
n = len(GROUPS)
fig = make_subplots(rows=n, cols=1, shared_xaxes=False, vertical_spacing=0.04,
    subplot_titles=GROUPS)

for i, g in enumerate(GROUPS, 1):
    gd = test[test['species'] == g].sort_values('date')
    fig.add_trace(go.Scatter(x=gd['date'], y=gd['mushroom_count'],
        mode='lines', name='Факт', line=dict(color=COLORS[g], width=2),
        showlegend=(i==1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=gd['date'], y=gd['predicted'],
        mode='lines', name='Прогноз', line=dict(color='black', width=2, dash='dot'),
        showlegend=(i==1)), row=i, col=1)

fig.update_layout(height=250*n, plot_bgcolor='white',
    title=f'Факт vs Прогноз — тест {TEST_YEAR} (только сезон каждой группы)',
    legend=dict(orientation='h', y=-0.03))
fig.update_yaxes(gridcolor='#e0e0e0')
fig.show()

## 3. Факт vs Прогноз — недельная агрегация (тест)

In [ ]:
fig = make_subplots(rows=n, cols=1, shared_xaxes=False, vertical_spacing=0.04,
    subplot_titles=GROUPS)

for i, g in enumerate(GROUPS, 1):
    gd = test[test['species'] == g].sort_values('date').set_index('date')
    fact_w = gd['mushroom_count'].resample('W').sum()
    pred_w = gd['predicted'].resample('W').sum()
    fig.add_trace(go.Scatter(x=fact_w.index, y=fact_w.values,
        mode='lines', name='Факт', line=dict(color=COLORS[g], width=2),
        showlegend=(i==1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=pred_w.index, y=pred_w.values,
        mode='lines', name='Прогноз', line=dict(color='black', width=2, dash='dot'),
        showlegend=(i==1)), row=i, col=1)

fig.update_layout(height=250*n, plot_bgcolor='white',
    title=f'Факт vs Прогноз — недельная агрегация (тест {TEST_YEAR})',
    legend=dict(orientation='h', y=-0.03))
fig.update_yaxes(gridcolor='#e0e0e0')
fig.show()

## 4. Факт vs Прогноз — трейн + тест (полный ряд)

In [ ]:
fig = make_subplots(rows=n, cols=1, shared_xaxes=True, vertical_spacing=0.03,
    subplot_titles=GROUPS)

for i, g in enumerate(GROUPS, 1):
    gd_tr = train[train['species'] == g].sort_values('date')
    gd_te = test[test['species'] == g].sort_values('date')
    
    fig.add_trace(go.Scatter(x=gd_tr['date'], y=gd_tr['mushroom_count'],
        mode='lines', name='Факт (трейн)', line=dict(color='#90CAF9', width=1), opacity=0.7,
        showlegend=(i==1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=gd_tr['date'], y=gd_tr['predicted'],
        mode='lines', name='Прогноз (трейн)', line=dict(color='#EF9A9A', width=1, dash='dot'), opacity=0.7,
        showlegend=(i==1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=gd_te['date'], y=gd_te['mushroom_count'],
        mode='lines', name='Факт (тест)', line=dict(color=COLORS[g], width=2),
        showlegend=(i==1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=gd_te['date'], y=gd_te['predicted'],
        mode='lines', name='Прогноз (тест)', line=dict(color='black', width=2, dash='dot'),
        showlegend=(i==1)), row=i, col=1)

fig.update_layout(height=220*n, plot_bgcolor='white',
    title='Факт vs Прогноз — полный ряд (трейн + тест)',
    legend=dict(orientation='h', y=-0.03))
fig.update_yaxes(gridcolor='#e0e0e0')
fig.show()

## 5. Калибровка (scatter: факт vs прогноз)

In [ ]:
cols = min(len(GROUPS), 3)
rows_n = (len(GROUPS) + cols - 1) // cols
fig = make_subplots(rows=rows_n, cols=cols, subplot_titles=GROUPS)

for idx, g in enumerate(GROUPS):
    r, c = divmod(idx, cols)
    gd = test[test['species'] == g]
    fig.add_trace(go.Scatter(
        x=gd['mushroom_count'], y=gd['predicted'],
        mode='markers', marker=dict(color=COLORS[g], size=4, opacity=0.5),
        name=g, showlegend=False,
    ), row=r+1, col=c+1)
    max_val = max(gd['mushroom_count'].max(), gd['predicted'].max(), 1)
    fig.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val],
        mode='lines', line=dict(color='#ccc', dash='dash', width=1),
        showlegend=False), row=r+1, col=c+1)
    fig.update_xaxes(title_text='Факт', row=r+1, col=c+1)
    fig.update_yaxes(title_text='Прогноз', row=r+1, col=c+1)

fig.update_layout(height=350*rows_n, plot_bgcolor='white',
    title='Калибровка: факт vs прогноз (тест)')
fig.update_yaxes(gridcolor='#e0e0e0')
fig.show()

## 6. Feature Importance (по каждой модели)

In [ ]:
cols = min(len(models), 3)
rows_n = (len(models) + cols - 1) // cols
fig = make_subplots(rows=rows_n, cols=cols, subplot_titles=list(models.keys()))

for idx, (g, model) in enumerate(models.items()):
    r, c = divmod(idx, cols)
    imp = pd.Series(
        model.feature_importance(importance_type='gain'),
        index=model.feature_name()
    ).sort_values(ascending=False).head(15)
    
    fig.add_trace(go.Bar(
        x=imp.values, y=imp.index,
        orientation='h', marker_color=COLORS.get(g, '#999'),
        name=g, showlegend=False,
    ), row=r+1, col=c+1)

fig.update_layout(height=400*rows_n, plot_bgcolor='white',
    title='Feature Importance (gain, top-15 для каждой модели)')
fig.update_xaxes(gridcolor='#e0e0e0')
fig.show()

## 7. MAE по месяцам

In [ ]:
test_err = test.copy()
test_err['error'] = test_err['predicted'] - test_err['mushroom_count']
test_err['abs_error'] = test_err['error'].abs()
test_err['month'] = test_err['date'].dt.month

cols = min(len(GROUPS), 3)
rows_n = (len(GROUPS) + cols - 1) // cols
fig = make_subplots(rows=rows_n, cols=cols, subplot_titles=GROUPS)

for idx, g in enumerate(GROUPS):
    r, c = divmod(idx, cols)
    gd = test_err[test_err['species'] == g]
    mae_m = gd.groupby('month')['abs_error'].mean().round(1)
    fig.add_trace(go.Bar(x=mae_m.index, y=mae_m.values,
        marker_color=COLORS[g], text=mae_m.values, textposition='outside',
        showlegend=False), row=r+1, col=c+1)
    fig.update_xaxes(dtick=1, row=r+1, col=c+1)

fig.update_layout(height=350*rows_n, plot_bgcolor='white',
    title='MAE по месяцам (тест)')
fig.update_yaxes(gridcolor='#e0e0e0')
fig.show()

## 8. Ошибка vs масштаб

In [ ]:
cols = min(len(GROUPS), 3)
rows_n = (len(GROUPS) + cols - 1) // cols
fig = make_subplots(rows=rows_n, cols=cols, subplot_titles=GROUPS)

for idx, g in enumerate(GROUPS):
    r, c = divmod(idx, cols)
    gd = test_err[test_err['species'] == g]
    fig.add_trace(go.Scatter(
        x=gd['mushroom_count'], y=gd['error'],
        mode='markers', marker=dict(color=COLORS[g], size=3, opacity=0.4),
        showlegend=False), row=r+1, col=c+1)
    fig.add_hline(y=0, line_dash='dash', line_color='black', row=r+1, col=c+1)

fig.update_layout(height=350*rows_n, plot_bgcolor='white',
    title='Ошибка (прогноз - факт) vs Факт')
fig.update_xaxes(title_text='Факт')
fig.update_yaxes(gridcolor='#e0e0e0', title_text='Ошибка')
fig.show()

## 9. Распределение ошибок + bias

In [ ]:
cols = min(len(GROUPS), 3)
rows_n = (len(GROUPS) + cols - 1) // cols
fig = make_subplots(rows=rows_n, cols=cols, subplot_titles=GROUPS)

for idx, g in enumerate(GROUPS):
    r, c = divmod(idx, cols)
    gd = test_err[test_err['species'] == g]
    fig.add_trace(go.Histogram(x=gd['error'], nbinsx=50,
        marker_color=COLORS[g], showlegend=False), row=r+1, col=c+1)
    fig.add_vline(x=0, line_dash='dash', line_color='black', row=r+1, col=c+1)

fig.update_layout(height=350*rows_n, plot_bgcolor='white',
    title='Распределение ошибок (прогноз - факт)')
fig.update_yaxes(gridcolor='#e0e0e0')
fig.show()

print('Bias (средняя ошибка) по группам:')
for g in GROUPS:
    gd = test_err[test_err['species'] == g]
    bias = gd['error'].mean()
    print(f'  {g:30s}: {bias:+.2f} ({"переоценка" if bias > 0 else "недооценка"})')

## 10. Топ дней с наибольшей ошибкой

In [ ]:
for g in GROUPS:
    gd = test_err[test_err['species'] == g].nlargest(5, 'abs_error')
    print(f'=== {g} ===')
    for _, row in gd.iterrows():
        print(f"  {row['date'].strftime('%d.%m.%Y')}: факт={row['mushroom_count']:.1f}, "
              f"прогноз={row['predicted']:.1f}, ошибка={row['error']:+.1f}")
    print()

## 11. Суммарный сезонный сбор: факт vs прогноз по годам

In [ ]:
preds_y = preds.copy()
preds_y['year'] = preds_y['date'].dt.year

cols = min(len(GROUPS), 3)
rows_n = (len(GROUPS) + cols - 1) // cols
fig = make_subplots(rows=rows_n, cols=cols, subplot_titles=GROUPS)

for idx, g in enumerate(GROUPS):
    r, c = divmod(idx, cols)
    gd = preds_y[preds_y['species'] == g]
    yf = gd.groupby('year')['mushroom_count'].sum()
    yp = gd.groupby('year')['predicted'].sum()
    fig.add_trace(go.Bar(x=yf.index.astype(str), y=yf.values,
        marker_color=COLORS[g], name='Факт', showlegend=(idx==0)), row=r+1, col=c+1)
    fig.add_trace(go.Bar(x=yp.index.astype(str), y=yp.values,
        marker_color='#333', opacity=0.4, name='Прогноз', showlegend=(idx==0)), row=r+1, col=c+1)

fig.update_layout(height=350*rows_n, plot_bgcolor='white', barmode='group',
    title='Суммарный сбор по годам: факт vs прогноз',
    legend=dict(orientation='h', y=-0.1))
fig.update_yaxes(gridcolor='#e0e0e0')
fig.show()

## 12. Сезонный профиль: факт vs прогноз (тест)

In [ ]:
test_w = test.copy()
test_w['week'] = test_w['date'].dt.isocalendar().week.astype(int)

cols = min(len(GROUPS), 3)
rows_n = (len(GROUPS) + cols - 1) // cols
fig = make_subplots(rows=rows_n, cols=cols, subplot_titles=GROUPS)

for idx, g in enumerate(GROUPS):
    r, c = divmod(idx, cols)
    gd = test_w[test_w['species'] == g]
    wf = gd.groupby('week')['mushroom_count'].mean()
    wp = gd.groupby('week')['predicted'].mean()
    fig.add_trace(go.Scatter(x=wf.index, y=wf.values,
        mode='lines', name='Факт', line=dict(color=COLORS[g], width=2),
        showlegend=(idx==0)), row=r+1, col=c+1)
    fig.add_trace(go.Scatter(x=wp.index, y=wp.values,
        mode='lines', name='Прогноз', line=dict(color='black', width=2, dash='dot'),
        showlegend=(idx==0)), row=r+1, col=c+1)
    fig.update_xaxes(title_text='Неделя', dtick=4, row=r+1, col=c+1)

fig.update_layout(height=350*rows_n, plot_bgcolor='white',
    title=f'Сезонный профиль: факт vs прогноз (тест {TEST_YEAR})',
    legend=dict(orientation='h', y=-0.1))
fig.update_yaxes(gridcolor='#e0e0e0')
fig.show()

## 13. Сравнение feature importance между группами

In [ ]:
# Топ-10 фич каждой модели — какие фичи важны для разных групп?
all_top = {}
for g, model in models.items():
    imp = pd.Series(
        model.feature_importance(importance_type='gain'),
        index=model.feature_name()
    ).sort_values(ascending=False)
    all_top[g] = imp.head(10).index.tolist()

print('Топ-10 фич для каждой группы:\n')
for g, feats in all_top.items():
    print(f'{g}:')
    for i, f in enumerate(feats, 1):
        # Отметим фичи уникальные для этой группы
        in_others = sum(1 for og, of in all_top.items() if og != g and f in of)
        marker = '' if in_others > 0 else ' ***UNIQUE***'
        print(f'  {i:2d}. {f}{marker}')
    print()

# Общие фичи
from collections import Counter
all_feats = Counter()
for feats in all_top.values():
    all_feats.update(feats)
print('Фичи, важные для нескольких групп:')
for f, cnt in all_feats.most_common(10):
    groups_with = [g for g, feats in all_top.items() if f in feats]
    print(f'  {f:35s} [{cnt} групп]: {", ".join(groups_with)}')